# PROJECT 5
by Szymon Waliczek
 - Impact of **Andreev** states caused by a superconducting - SC drain lead  on 2DEG electron transport in **InAs**
 - Normal wire with leads
 - But the drain electrode is **SC**
 - **No spin**
 - **No Peierls phase**

In [ ]:
import ipyparallel as ipp
cluster = ipp.Client(profile="kwant_parallel")
v = cluster[:]
lview = cluster.load_balanced_view()
len(v)

In [ ]:
%%px --local

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
%%px --local

import kwant
import tinyarray
import numpy as np

s_x = tinyarray.array([[0, 1], [1, 0]])
s_y = tinyarray.array([[0, -1j], [1j, 0]])
s_z = tinyarray.array([[1, 0], [0, -1]])

In [ ]:
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

In [ ]:
%%px --local

# Physical constants
from scipy.constants import physical_constants
eV = physical_constants['electron volt'][0]
m_el = physical_constants['electron mass'][0]
h_bar = physical_constants['Planck constant over 2 pi'][0]
a = 5
m_eff = 0.023 * m_el
t = (h_bar**2 / (2 * m_eff * (a*1e-9)**2)) / eV # hopping in eV
W, L = 100, 200
barrierpos = (85, 95) # 10nm width
Deltapos = 95
freedom_deg = 2; # e+ holes
PI = np.pi

In [ ]:
print(f"h_bar = {h_bar}")
print(f"m_eff = {m_eff}")
print(f"a = {a} nm")
print(f"t = {t} eV")
print(f"W, L = {W, L} nm")

In [ ]:
%%px --local

def onsite(site, mu, delta, barrier):
    (x, y) = site.pos
    if barrierpos[0] <= y < barrierpos[1]: 
        return (4*t + barrier - mu)*s_z
    if y >= Deltapos:
        return (4*t - mu)*s_z + delta*s_x
    else:
        return (4*t - mu)*s_z
    
def hop(site1, site2):
    return -t * s_z

def lead_onsite(site, mu, delta, barrier):
    return (4 * t - mu) * s_z

def lead_onsite_SC(site, mu, delta, barrier):
    return (4 * t - mu) * s_z + delta * s_x

In [ ]:
%%px --local

def rectangle(pos, width, length):
    (x, y) = pos
    return abs(x) < width/2 and abs(y) < length/2
#-----------------------------------------------------------------------------
def lead_shape(pos, width):
    return abs(pos[0]) < width / 2
#-----------------------------------------------------------------------------
def make_sys_SC(a, width, length):
    lat = kwant.lattice.square(a, norbs=2) 
    sys = kwant.Builder()
    sys[lat.shape(lambda pos: rectangle(pos, width, length), (0, 0))] = onsite
    sys[lat.neighbors(1)] = hop
    sym_down = kwant.TranslationalSymmetry((0, -a))
    lead_down = kwant.Builder(sym_down, conservation_law=-s_z, particle_hole=s_y)
    lead_down[lat.shape(lambda pos: lead_shape(pos, width), (0, 0))] = lead_onsite
    lead_down[lat.neighbors()] = hop
    sym_up = kwant.TranslationalSymmetry((0, a))
    lead_up = kwant.Builder(sym_up)
    lead_up[lat.shape(lambda pos: lead_shape(pos, width), (0, 0))] = lead_onsite_SC
    lead_up[lat.neighbors()] = hop
    sys.attach_lead(lead_down)
    sys.attach_lead(lead_up)
    return sys.finalized()

In [ ]:
def plot_sys(sys):
    kwant.plot(sys, fig_size=(2, 3), show=False)
    plt.title(f"fsys a = {a}nm") 
    plt.xlabel("x [nm]")
    plt.ylabel("y [nm]")
    plt.show()

In [ ]:
%%px --local

fsys = make_sys_SC(a, W, L)

In [ ]:
plot_sys(fsys)

In [ ]:
%%px --local

def compute_conductance(energy, sys, params):
    smatrix = kwant.smatrix(sys, energy, params=params)
    n_channels = smatrix.submatrix((0, 0), (0, 0)).shape[0]
    ree = smatrix.transmission((0, 0), (0, 0))
    rhe = smatrix.transmission((0, 1), (0, 0))
    return n_channels - ree + rhe

def compute_density_parallel(E, sys, params):
    modes = sys.leads[0].modes(energy=E, params=params)[0]
    channels = len(modes.momenta) // 2
    print(f"For energy = {E} | {channels} open channels")
    if channels == 0: 
        return None
    else:
        res = lview.map_sync(lambda n: kwant.operator.Density(sys)(kwant.wave_function(
                                        sys, E, params=params)(0)[n]), range(channels))
        return sum(res)

In [ ]:
def plot_bands(lead, ymin, ymax, xlim, params):
    bands = kwant.physics.Bands(lead, params=params)
    momenta = np.linspace(-xlim / (a), xlim / (a), 500)
    energies = np.array([bands(k) for k in momenta])    
    plt.figure(figsize=(3, 3))
    for i in range(energies.shape[1]):
        # Sprawdzamy energię w środku zakresu (k=0), aby odróżnić typ paraboli
        # e > 0 to zazwyczaj elektron (niebieski), e < 0 to dziura (czerwony)
        color = 'blue' if energies[len(momenta)//2, i] > 0 else 'red'        
        plt.plot(momenta, energies[:, i], color=color)
    plt.ylabel("$E [eV]$")
    plt.xlabel("$k_y$ [1/nm]")
    plt.ylim(ymin, ymax)
    plt.grid()
    plt.show()
    
def plot_conductance(energy, params):
    transmission = lview.map_sync(lambda e: compute_conductance(e, fsys, params), energy)
    plt.figure(figsize=(3.4, 3))
    plt.plot(energy, transmission, lw=1)
    plt.xlabel("$E [eV]$")
    plt.ylabel("$G [e^2/h]$")
    plt.grid(True)
    plt.show()
    
def plot_LDOS(sys, E, params):
    psi2 = compute_density_parallel(E, sys, params=params)
    if psi2 is not None:
        kwant.plotter.map(sys, psi2, show=False, fig_size=(4, 3), vmin=0, vmax=max(psi2))
        plt.title(fr"$|\Psi|^2$ | $E = {E}$ eV")
        plt.xlabel("x [nm]"); plt.ylabel("y [nm]")
        plt.show()

# Stare przewodnictwo:?????-----------------------
"""def compute_G_old(energy, sys, p):
    smatrix = kwant.smatrix(sys, energy, params=p)
    n_channels = smatrix.submatrix((0, 0), (0, 0)).shape[0]
    ree = smatrix.transmission((0, 0), (0, 0))
    rhe = smatrix.transmission((0, 1), (0, 0))
    return n_channels - ree + rhe

def plot_G_old(energy, name, sys, p):
    transmission = lview.map_sync(lambda e: compute_G_old(e, sys, p), energy)
    plt.figure(figsize=(3.4, 3))
    plt.plot(energy*1e3, transmission, lw=1)
    plt.title(f"{name} | $\mu = {p['mu']*1e3}$ meV | B = {p['B']}$ T | \delta = {p['delta']*1e3}$ meV")
    plt.axvline(p['delta']*1e3, color='black', linestyle=':', label='$\Delta$', lw=1)
    plt.axvline(p['mu']*1e3, color='green', linestyle=':', label='$\mu$', lw=1)
    plt.axvline(-p['delta']*1e3, color='black', linestyle=':', lw=1)
    plt.legend(loc='upper right', fontsize=8)
    plt.xlabel("$E [meV]$")
    plt.ylabel("$G [e^2/h]$")
    plt.grid(True)
    plt.show()"""
#-------------------------------------------------------------------------------------------------------------------

In [ ]:
def plot_mu_conductance(sys, mu, barrier):
    g_normal = []
    g_andreev = []
    
    # Stałe parametry - liczymy dla E=0
    params_norm = dict(delta=0, barrier=barrier)
    params_andreev = dict(delta=0.0001, barrier=barrier)
    
    for i in mu:
        params_norm['mu'] = i
        params_andreev['mu'] = i
        g_normal.append(compute_conductance(0, sys, params_norm))
        g_andreev.append(compute_conductance(0, sys, params_andreev))

    plt.figure(figsize=(3, 3))
    plt.plot(mu, g_normal, label='Normal $\Delta=0, G=1*N$', color='blue')
    plt.plot(mu, g_andreev, label='Andreev $\Delta>0, G=2*N$', color='red')
    plt.xlabel("$\mu$ [eV]")
    plt.ylabel("G [$e^2/h$]")
    plt.legend(fontsize=6)
    plt.grid(True); 
    plt.show()

In [ ]:
# 1. Tunneling
params_1 = dict(mu=0.003, delta=0.001, barrier=0.01)
Evals_1 = np.linspace(-2.5*0.001, 2.5*0.001, 200)

plot_bands(fsys.leads[1], -0.002, 0.002, PI, params_1)
plot_conductance(Evals_1, params_1)
plot_LDOS(fsys, E=0, params=params_1)

In [ ]:
# 2. Andreev Plateau
params_2 = dict(mu=0.003, delta=0.001, barrier=0)
Evals_2 = np.linspace(-2.5*0.0001, 2.5*0.0001, 200)

plot_bands(fsys.leads[0], -0.002, 0.002, PI, params_2)
plot_conductance(Evals_2, params_2)
plot_LDOS(fsys, E=0, params=params_2)

In [ ]:
mu = np.linspace(-0.001, 0.005, 20)
plot_mu_conductance(fsys, mu, barrier=0.0) #od baeriery a nie mu

**Efect of Δ modulation on LDOS**

In [ ]:
from ipywidgets import interact
import ipywidgets as widgets

In [ ]:
def interactive_LDOS(mu, delta, barrier):
    current_params = dict(mu=mu, delta=delta, barrier=barrier)
    plot_LDOS(fsys, E=0, params=current_params)
def interactive_conductance(mu, delta, barrier):
    energy = np.linspace(-2.5*delta, 2.5*delta, 200)
    current_params = dict(mu=mu, delta=delta, barrier=barrier)
    plot_conductance(energy=energy, params=current_params)

In [ ]:
interact(
    interactive_LDOS, 
    mu=widgets.FloatSlider(min=0, max=0.4, step=0.001, value=0.003, description='mu:', readout_format='.3f'),
    delta=widgets.FloatSlider(min=-0.01, max=0.01, step=0.0001, value=0.0001, description='delta:', readout_format='.4f'),
    barrier=widgets.FloatSlider(min=0, max=1.5, step=0.01, value=0, description='barrier:', readout_format='.2f')
);

In [ ]:
interact(
    interactive_conductance,
    mu=widgets.FloatSlider(min=0, max=0.53, step=0.001, value=0.003, description='mu:', readout_format='.3f'),
    delta=widgets.FloatSlider(min=-0.01, max=0.01, step=0.0001, value=0.0001, description='delta:', readout_format='.4f'),
    barrier=widgets.FloatSlider(min=0, max=1.5, step=0.01, value=0, description='barrier:', readout_format='.2f')
);